In [0]:
_checkpoints = "dbfs:/Volumes/workspace/retails/raw_data/_checkpoints/retails/dev/silver/order_items_cleaned"

In [0]:
from pyspark.sql.functions import to_date, current_date, col

# Step-1: Read from Bronze (Streaming)
# 
cdc_df = spark.readStream.table("retails.silver.order_items_cdc")


In [0]:
from pyspark.sql.functions import to_json, struct, when, col

# Step 4: Apply Type Casting & Standardization
silver_df = cdc_df.withColumn("order_item_id", col("order_item_id").cast("bigint")) \
                    .withColumn("order_item_order_id", col("order_item_order_id").cast("bigint")) \
                    .withColumn("order_item_product_id", col("order_item_product_id").cast(("bigint"))) \
                    .withColumn("order_item_quantity", col("order_item_quantity").cast("bigint")) \
                    .withColumn("order_item_subtotal", col("order_item_subtotal").cast("decimal(10,2)")) \
                    .withColumn("order_item_product_price", col("order_item_product_price").cast("decimal(10,2)")) \
                    .withColumn("record_hash", col("record_hash")) \
                    .withColumn("batch_id", col("batch_id").cast("string"))
                        


In [0]:
# make ready to upsert silver table order_items_cleaned
# silver_df = cdc_df \
#             .withColumn("product_id", col("product_id").cast("bigint")) \
#             .withColumn("product_price", col("product_price").cast("decimal(10,2)")) \
#             .withColumn("product_category_id", col("product_category_id").cast("bigint")) \
#             .withColumn("batch_id", col("batch_id").cast("string"))


In [0]:
# upsert order_items_cleaned table
MERGE_UPDATE = """
    MERGE INTO retails.silver.order_items_cleaned t
    USING order_items_cleaned_vw s
    ON t.order_item_id = s.order_item_id
    WHEN MATCHED AND s.record_hash != t.record_hash AND s.op='UPDATE' THEN
        UPDATE SET t.order_item_order_id = s.order_item_order_id, t.order_item_product_id = s.order_item_product_id, t.order_item_quantity = s.order_item_quantity, t.order_item_subtotal = s.order_item_subtotal, t.order_item_product_price = s.order_item_product_price, t.is_deleted = false, t.updated_ts = current_timestamp(), t.op= s.op, t.record_hash = s.record_hash
    
    WHEN MATCHED AND s.is_deleted AND s.op='DELETE' THEN
        UPDATE SET t.is_deleted = true, t.updated_ts = current_timestamp(), t.op= s.op
    
    WHEN NOT MATCHED AND s.op='INSERT' THEN
        INSERT (
            order_item_id,
            order_item_order_id,
            order_item_product_id,
            order_item_quantity,
            order_item_subtotal,
            order_item_product_price,
            is_deleted,
            ingestion_ts,
            ingestion_dt,
            source_system,
            source_file_name,
            batch_id,
            run_id,
            op,
            record_hash,
            created_ts,
            updated_ts
                    )
        VALUES(
            s.order_item_id,
            s.order_item_order_id,
            s.order_item_product_id,
            s.order_item_quantity,
            s.order_item_subtotal,
            s.order_item_product_price,
            s.is_deleted,
            s.ingestion_ts,
            s.ingestion_dt,
            s.source_system,
            s.source_file_name,
            s.batch_id,
            s.run_id,
            s.op,
            s.record_hash,
            current_timestamp(),
            current_timestamp()
            )
    """

# spark.sql(merge_query).show()



In [0]:
def upsert_to_silver(batch_df, batch_id):

    # batch_df = batch_df.cache()
    batch_df.createOrReplaceTempView(
        "order_items_cleaned_vw"
    )
    spark.sql(MERGE_UPDATE)

    # batch_df.unpersist()

In [0]:
silver_df = silver_df.drop("event_ts")
# silver_df.printSchema()

In [0]:

silver_df.writeStream \
    .format("delta") \
    .foreachBatch(upsert_to_silver) \
    .option("checkpointLocation", _checkpoints) \
    .trigger(once=True) \
    .start() \
    .awaitTermination()

In [0]:
# quarantine_status	Meaning
# NEW	Newly quarantined record, not yet reviewed
# INVALID	Permanently invalid record
# RESCUED	Record has rescued_data but may be recoverable
# CORRUPT	Completely corrupt JSON/file structure
# PARTIAL_VALID	Some columns valid, some problematic
# FIXED	Data corrected successfully
# REPROCESSED	Re-ingested into clean table

In [0]:
# GRANTS = "ALTER TABLE retails.silver.order_items_quarantine OWNER TO `dp-sales-engineers`"
# spark.sql(GRANTS).display()